In [ ]:
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
super_ai_engineer_ss_6_sleep_stage_classification_path = kagglehub.competition_download('super-ai-engineer-ss-6-sleep-stage-classification')

100%|██████████| 2.13G/2.13G [00:26<00:00, 85.3MB/s]

Extracting files...


# **Preparation Data**

In [ ]:
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import numpy as np
import joblib
import random
import glob
import os

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.utils import to_categorical

In [ ]:

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import skew, kurtosis, iqr, entropy


In [ ]:
path = "/root/.cache/kagglehub/competitions/super-ai-engineer-ss-6-sleep-stage-classification/train/train"
csv_files = sorted(glob.glob(f"{path}/train*.csv"))
df_list = [pd.read_csv(file) for file in csv_files]
df = pd.concat(df_list, ignore_index=True)

In [ ]:
print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32037600 entries, 0 to 32037599
Data columns (total 9 columns):
 #   Column       Dtype  
---  ------       -----  
 0   BVP          float64
 1   ACC_X        float64
 2   ACC_Y        float64
 3   ACC_Z        float64
 4   TEMP         float64
 5   EDA          float64
 6   HR           float64
 7   IBI          float64
 8   Sleep_Stage  object 
dtypes: float64(8), object(1)
memory usage: 2.1+ GB
None
         BVP      ACC_X      ACC_Y     ACC_Z       TEMP       EDA         HR  \
0  25.325870 -21.809247 -60.302750  4.940839  31.722653  0.064595  72.015570   
1  20.021505 -19.437787 -60.565345  7.408788  31.722647  0.064523  72.015802   
2  16.314478 -21.624667 -61.142561  4.105717  31.722735  0.064659  72.017417   
3   9.324392 -21.761314 -61.985822  3.972967  31.722564  0.064440  72.013801   
4  -1.014338 -19.055301 -59.934137  9.097628  31.722790  0.065397  72.018920   

        IBI Sleep_Stage  
0  1.050338           W  
1  1.05033

In [ ]:
window_size = 480

def process_segments(df, window_size):
    num_segments = len(df) // window_size
    processed_data = []

    for i in tqdm(range(num_segments), desc="Processing Segments", unit="window"):
        start_idx = i * window_size
        end_idx = (i + 1) * window_size
        segment = df.iloc[start_idx:end_idx]

        feature_dict = {}

        for col in df.columns[:-1]:
            data = segment[col].values

            feature_dict[f'{col}_mean'] = np.mean(data)
            feature_dict[f'{col}_std'] = np.std(data)
            feature_dict[f'{col}_mad'] = np.median(np.abs(data - np.median(data)))
            feature_dict[f'{col}_max'] = np.max(data)
            feature_dict[f'{col}_min'] = np.min(data)
            feature_dict[f'{col}_sma'] = np.sum(np.abs(data))
            feature_dict[f'{col}_energy'] = np.sum(np.square(data))
            feature_dict[f'{col}_iqr'] = iqr(data)
            feature_dict[f'{col}_skewness'] = skew(data)
            feature_dict[f'{col}_kurtosis'] = kurtosis(data)

        feature_dict['Sleep_Stage'] = segment.iloc[0, -1]

        processed_data.append(feature_dict)

    reduced_df = pd.DataFrame(processed_data)
    return reduced_df

reduced_df = process_segments(df, window_size)

print(f"Original shape: {df.shape}")
print(f"Reduced shape: {reduced_df.shape}")


Processing Segments:   0%|          | 0/66745 [00:00<?, ?window/s]/tmp/ipykernel_2187/1246149283.py:25: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  feature_dict[f'{col}_skewness'] = skew(data)
/tmp/ipykernel_2187/1246149283.py:26: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  feature_dict[f'{col}_kurtosis'] = kurtosis(data)
Processing Segments: 100%|██████████| 66745/66745 [20:42<00:00, 53.71window/s]


Original shape: (32037600, 9)
Reduced shape: (66745, 81)


In [ ]:
print(reduced_df.head())

   BVP_mean     BVP_std    BVP_mad     BVP_max      BVP_min       BVP_sma  \
0  2.816168  155.888299  16.669035  665.980817 -1212.411684  36277.202258   
1 -2.661875  123.138367  34.460027  466.121232  -729.854625  33042.523995   
2 -0.244761   52.886585  25.143280  154.234163  -184.643390  18139.189771   
3  0.004561   76.540402  41.783889  229.873620  -344.279599  27405.357588   
4 -0.944100  153.643422  44.680301  583.380258  -552.651515  45599.250622   

     BVP_energy    BVP_iqr  BVP_skewness  BVP_kurtosis  ...       IBI_std  \
0  1.166836e+07  45.494987     -2.017460     14.686670  ...  4.305011e-02   
1  7.281669e+06  67.349743     -1.671969      9.710478  ...  4.752380e-14   
2  1.342584e+06  49.991540     -0.552818      1.478818  ...  6.217268e-02   
3  2.812048e+06  82.239137     -0.829661      2.316443  ...  1.147790e-14   
4  1.133145e+07  92.415518      0.077832      3.473697  ...  4.620177e-02   

        IBI_mad   IBI_max   IBI_min     IBI_sma  IBI_energy       IBI_iqr 

# **Model**

In [ ]:
X = reduced_df.drop(columns=['Sleep_Stage'])
y = reduced_df['Sleep_Stage']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train Labels: {y_train.value_counts()}")
print(f"Test Labels: {y_test.value_counts()}")
print(f"Train shape: {X_train.shape}, y shape: {y_train.shape}")

Train shape: (53396, 80), Test shape: (13349, 80)
Train Labels: Sleep_Stage
N2    27029
W     12662
N1     6203
R      5626
N3     1876
Name: count, dtype: int64
Test Labels: Sleep_Stage
N2    6757
W     3166
N1    1550
R     1407
N3     469
Name: count, dtype: int64
Train shape: (53396, 80), y shape: (53396,)


In [ ]:
print(X_train.isnull().sum().sum())
print(X_train.isnull().sum())

8063
BVP_mean           0
BVP_std            0
BVP_mad            0
BVP_max            0
BVP_min            0
                ... 
IBI_sma            0
IBI_energy         0
IBI_iqr            0
IBI_skewness    2318
IBI_kurtosis    2318
Length: 80, dtype: int64


In [ ]:
X_train.fillna(X_train.median(), inplace=True)
X_test.fillna(X_test.median(), inplace=True)

In [ ]:
rf = RandomForestClassifier(random_state=42, class_weight='balanced')

param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

random_search = RandomizedSearchCV(rf, param_distributions=param_grid,n_iter=5, cv=3, scoring='accuracy',verbose=2, n_jobs=-1, random_state=42)

for _ in tqdm(range(random_search.n_iter), desc="RandomizedSearchCV Iterations"):
    random_search.fit(X_train, y_train)

best_rf = random_search.best_estimator_
print(f"Best Parameters: {random_search.best_params_}")

# **Evaluation**

In [ ]:
y_pred = best_rf.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")


In [ ]:
importances = best_rf.feature_importances_

indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("Feature Importance")
plt.barh(range(X_train.shape[1]), importances[indices], align="center")
plt.yticks(range(X_train.shape[1]), [X_train.columns[i] for i in indices])
plt.xlabel("Importance")
plt.show()


In [ ]:
joblib.dump(best_rf, 'best_random_forest_model.pkl')


# **Submission**

In [ ]:
%ls /root/.cache/kagglehub/competitions/super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/

In [ ]:
test_path = "/root/.cache/kagglehub/competitions/super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/"

test_folders = sorted(os.listdir(test_path))

test_data = []

for folder in test_folders:
    folder_path = os.path.join(test_path, folder)

    csv_files = sorted(os.listdir(folder_path))

    for file in csv_files:
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)

        df['ID'] = file.replace(".csv", "")

        test_data.append(df)

test_df = pd.concat(test_data, ignore_index=True)

print(f"Total Test Data Shape: {test_df.shape}")
(test_df.head())


In [ ]:
test_reduced_df = process_segments(test_df, window_size=480)
print(f"Processed Test Data Shape: {test_reduced_df.shape}")
print(test_reduced_df.head())

In [ ]:
X_test = test_reduced_df.drop(columns=['Sleep_Stage'])
ID_column = test_reduced_df['Sleep_Stage'].rename('ID')
X_test.head()

In [ ]:
X_test = X_test.fillna(X_test.mean())

In [ ]:
X_test_final = X_test

test_predictions = best_rf.predict(X_test_final)

test_reduced_df["Predicted_Sleep_Stage"] = test_predictions
test_reduced_df.to_csv("/content/submission_sleep_stages.csv", index=False)

print("Predictions saved")

In [ ]:
test_reduced_df.rename(columns={'Sleep_Stage': 'id'}, inplace=True)
test_reduced_df.rename(columns={'Predicted_Sleep_Stage': 'labels'}, inplace=True)

In [ ]:
final_df = test_reduced_df[['id', 'labels']]
final_df.to_csv("/content/submission_sleep.csv", index=False)
print("CSV file saved as submission.csv")